# z-shift Tier B - GPU experiments

Everything in the protocol that needs CUDA: **B1-B8**. Tier A is CPU-only and is
not run here.

Runs on **Kaggle** (recommended) or **Colab**. Kaggle is preferred because
*Save & Run All* executes headless for up to 12 h and survives a closed browser,
and because a private Kaggle Dataset mounts your DTU / capture data read-only at
`/kaggle/input` every session with no re-download.

## Read this before you press Run All

1. **`--quick` is a no-op in Tier B.** `experiment_parser` defines the flag and
   not one of the eight `exp_b*.py` modules forwards it to `run()`. Typing the
   smoke-test command in a terminal silently launches the full grid. This
   notebook therefore never passes `--quick`; scope is controlled by the flags
   that are actually honoured (`--scenes`, `--n-images`, `--frame-counts`,
   `--thresholds`, `--no-budget-sweep`). Wiring `--quick` properly is still
   TODO P0 and worth doing.
2. **Run the SMOKE lane first.** It exercises every module end to end in
   minutes. A three-day run that dies at row 1 on a `KeyError` is the failure
   mode this exists to prevent.
3. **The repo is cloned from GitHub.** `bench/`, `paper/`,
   `instrumentation.py` and `tests/test_bench.py` are untracked on your laptop.
   Commit and push before running this, or the clone will not contain them.
4. **`data/` and `third_party/` are gitignored.** MASt3R is cloned and installed
   by this notebook; the dataset must come from a mounted Kaggle Dataset or
   Drive folder that you point `DATA_ROOT` at.

## What "proper error logs" means here

- One log file per experiment under `<results>/logs/<run_id>/`, plus a combined
  `session.log` holding every log record from the notebook itself.
- Each experiment runs in its **own subprocess**, so a CUDA OOM or a VTK
  segfault kills that experiment and not the kernel. Exit code, signal, wall
  time and the tail of the output are recorded either way.
- `faulthandler` is enabled in parent and children, so a native crash leaves a
  Python traceback instead of a silent death.
- `run_summary.json` is rewritten after **every** experiment, so a session
  killed at the 12 h wall still leaves a full account of what finished.
- Re-running the notebook **skips experiments whose CSV already exists**, so a
  killed session resumes rather than restarts.

## 1. Config

The only cell you should need to edit.

In [ ]:
# --- repo -----------------------------------------------------------------
REPO_URL = "https://github.com/AyushK0808/z-shift.git"
REPO_BRANCH = "main"

# --- data -----------------------------------------------------------------
# Kaggle: attach your dataset, then point this at its mount,
#   e.g. "/kaggle/input/zshift-tier-b"
# Colab: a Drive folder, e.g. "/content/drive/MyDrive/zshift-tier-b"
DATA_ROOT = "/kaggle/input/zshift-tier-b"

# Scene manifest, written fresh into the work dir with absolute paths so a
# read-only /kaggle/input mount is fine. Paths below are relative to DATA_ROOT.
# tau is the F-score threshold in the dataset's own units; DTU's convention is
# 2 mm. Label COLMAP pseudo-GT as such -- it goes in the paper.
SCENES = [
    {
        "name": "scan24",
        "image_dir": "dtu/scan24/images",
        "gt_path": "dtu/scan24/stl024_total.ply",
        "tau": 2.0,
        "units": "mm",
        "notes": "DTU eval subset; sensor ground truth",
    },
    {
        "name": "scan37",
        "image_dir": "dtu/scan37/images",
        "gt_path": "dtu/scan37/stl037_total.ply",
        "tau": 2.0,
        "units": "mm",
        "notes": "DTU eval subset; sensor ground truth",
    },
    # B4 needs an extracted video sequence in capture order with MORE than
    # MAX_RECONSTRUCTION_FRAMES (40) frames, or the budget never fires and all
    # three variants are identical.
    # {
    #     "name": "desk_orbit",
    #     "image_dir": "captures/desk_orbit/frames",
    #     "gt_path": "captures/desk_orbit/colmap_dense.ply",
    #     "tau": 0.01,
    #     "units": "scene units (COLMAP)",
    #     "notes": "pseudo-GT from COLMAP dense",
    # },
]

# --- scope ----------------------------------------------------------------
# "smoke" = minutes per experiment, validates the orchestration, NOT publishable.
# "full"  = the real grid.
SCOPE = "smoke"

# Which experiments to run, in order. Cheap-and-informative first, so a session
# that dies early still bought you something.
EXPERIMENTS = ["b7", "b2", "b8", "b6", "b3", "b5", "b4", "b1"]

# Per-experiment ceiling. A run that blows through this is a bug, not progress.
TIMEOUT_MIN = {
    "smoke": {"b1": 45, "b2": 30, "b3": 45, "b4": 120, "b5": 45, "b6": 30, "b7": 30, "b8": 30},
    "full": {"b1": 180, "b2": 120, "b3": 240, "b4": 480, "b5": 240, "b6": 120, "b7": 90, "b8": 120},
}

# Stop LAUNCHING new experiments once the session is this old (minutes).
# Kaggle kills at 12 h (720 min); leave room to collect results.
SESSION_BUDGET_MIN = 660

# Delete each experiment's reconstruction outputs after it finishes. The sparse
# alignment cache is ~1.5 GB per run and accumulates fast.
CLEAN_WORK_AFTER_EACH = True

# Force numpy < 2 to match pyproject. Needs a kernel restart and can break
# preinstalled wheels compiled against numpy 2 -- only set this if the import
# check in the preflight cell actually fails.
PIN_NUMPY_LT2 = False

# Re-run experiments whose CSV already exists.
FORCE_RERUN = False

## 2. Paths and logging

Sets up the log tree and installs handlers that catch tracebacks from notebook
cells, from `warnings`, and from native crashes.

In [ ]:
import datetime as _dt
import faulthandler
import json
import logging
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from collections import deque
from pathlib import Path

faulthandler.enable()


def _detect_platform():
    if Path("/kaggle/working").exists():
        return "kaggle"
    if "google.colab" in sys.modules or Path("/content").exists():
        return "colab"
    return "local"


PLATFORM = _detect_platform()

if PLATFORM == "kaggle":
    # /kaggle/working is persisted but capped (~20 GB) -- results only.
    # /kaggle/temp is ephemeral and roomy -- caches and reconstructions.
    OUT_DIR = Path("/kaggle/working/tier_b")
    WORK_DIR = Path("/kaggle/temp/zshift")
    REPO_DIR = Path("/kaggle/working/z-shift")
elif PLATFORM == "colab":
    drive = Path("/content/drive/MyDrive")
    OUT_DIR = (drive / "zshift-tier-b-results") if drive.exists() else Path("/content/tier_b")
    WORK_DIR = Path("/content/zshift-work")
    REPO_DIR = Path("/content/z-shift")
else:
    OUT_DIR = Path.cwd() / "tier_b_out"
    WORK_DIR = Path.cwd() / "tier_b_work"
    REPO_DIR = Path.cwd()

RESULTS_DIR = OUT_DIR / "results"
RUN_ID = _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_DIR = OUT_DIR / "logs" / RUN_ID
HF_HOME = WORK_DIR / "hf"

for d in (OUT_DIR, WORK_DIR, RESULTS_DIR, LOG_DIR, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)

SESSION_LOG = LOG_DIR / "session.log"
CRASH_LOG = LOG_DIR / "faulthandler.log"

# Held open for the whole session so a native crash has somewhere to dump.
_crash_fh = open(CRASH_LOG, "w")  # noqa: SIM115
faulthandler.enable(file=_crash_fh, all_threads=True)

log = logging.getLogger("tierb")
log.setLevel(logging.DEBUG)
log.handlers.clear()
log.propagate = False

_fmt = logging.Formatter("%(asctime)s %(levelname)-8s %(name)s: %(message)s", datefmt="%H:%M:%S")
_fh = logging.FileHandler(SESSION_LOG, encoding="utf-8")
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
log.addHandler(_fh)
log.addHandler(_sh)

logging.captureWarnings(True)
_warn_log = logging.getLogger("py.warnings")
_warn_log.handlers.clear()
_warn_log.addHandler(_fh)


def _log_uncaught(exc_type, exc, tb):
    log.error("uncaught exception", exc_info=(exc_type, exc, tb))


sys.excepthook = _log_uncaught

# IPython swallows sys.excepthook, so hook its traceback printer too. Without
# this, a cell that raises leaves nothing in session.log.
try:
    _ip = get_ipython()  # noqa: F821
except NameError:
    _ip = None
if _ip is not None and not getattr(_ip, "_tierb_hooked", False):
    _orig_showtraceback = _ip.showtraceback

    def _showtraceback(*args, **kwargs):
        log.error("cell raised", exc_info=sys.exc_info())
        return _orig_showtraceback(*args, **kwargs)

    _ip.showtraceback = _showtraceback
    _ip._tierb_hooked = True

log.info("platform=%s run_id=%s", PLATFORM, RUN_ID)
log.info("repo=%s", REPO_DIR)
log.info("work=%s (ephemeral: caches, reconstructions)", WORK_DIR)
log.info("out=%s (persisted: CSVs, logs)", OUT_DIR)
log.info("session log -> %s", SESSION_LOG)

## 3. Command runner

Streams output live, tees every line to a log file, caps how much reaches the
notebook (a 3 h run must not bloat the `.ipynb`), and turns a non-zero exit into
an exception carrying the tail of the output.

In [ ]:
import threading

MAX_ECHO_LINES = 400  # per command; the log file always gets everything


def describe_exit(rc):
    if rc == 0:
        return "ok"
    if rc == -9:
        return "SIGKILL -- almost always the OOM killer (host RAM, not VRAM)"
    if rc == -11:
        return "SIGSEGV -- native crash; check faulthandler.log for Python frames"
    if rc == -6:
        return "SIGABRT -- native abort (CUDA / VTK)"
    if rc < 0:
        return "killed by signal " + str(-rc)
    return "non-zero exit " + str(rc)


class CommandFailed(RuntimeError):
    def __init__(self, cmd, returncode, tail, log_path, timed_out=False, timeout_s=None):
        self.cmd = cmd
        self.returncode = returncode
        self.tail = tail
        self.log_path = log_path
        self.timed_out = timed_out
        self.timeout_s = timeout_s
        # A timeout kill also lands as SIGKILL on Linux. Say which one it was,
        # or every timeout gets misread as the OOM killer.
        if timed_out:
            self.reason = "timed out after %.1f min and was killed" % ((timeout_s or 0) / 60)
        else:
            self.reason = describe_exit(returncode)
        super().__init__(f"{self.reason}\n{tail}")


def sh(
    cmd, *, cwd=None, log_path=None, env=None, timeout_s=None, check=True, echo=True, tail_lines=60
):
    # Run cmd (list, or str via bash -lc), tee output to log_path,
    # return (returncode, tail_text).
    if isinstance(cmd, str):
        cmd = ["bash", "-lc", cmd]
    log_path = Path(log_path) if log_path else (LOG_DIR / "commands.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)

    full_env = dict(os.environ)
    if env:
        full_env.update({k: str(v) for k, v in env.items()})

    printable = " ".join(str(c) for c in cmd)
    header = (
        "\n"
        + "=" * 78
        + "\n$ "
        + printable
        + "\n  cwd="
        + str(cwd)
        + "  started="
        + _dt.datetime.now().isoformat(timespec="seconds")
        + "\n"
        + "=" * 78
        + "\n"
    )
    log.debug("running: %s", printable)

    tail = deque(maxlen=max(tail_lines, 200))
    echoed = [0]
    start = time.monotonic()

    with open(log_path, "a", encoding="utf-8", errors="replace") as fh:
        fh.write(header)
        fh.flush()
        proc = subprocess.Popen(  # noqa: S603
            [str(c) for c in cmd],
            cwd=str(cwd) if cwd else None,
            env=full_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors="replace",
        )

        def pump():
            for line in proc.stdout:
                fh.write(line)
                tail.append(line)
                if echo:
                    if echoed[0] < MAX_ECHO_LINES:
                        print(line, end="")
                        echoed[0] += 1
                    elif echoed[0] == MAX_ECHO_LINES:
                        print("... output continues in " + str(log_path))
                        echoed[0] += 1
            fh.flush()

        pumper = threading.Thread(target=pump, daemon=True)
        pumper.start()

        timed_out = False
        try:
            rc = proc.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired:
            timed_out = True
            log.error("TIMEOUT after %.1f min, killing: %s", (timeout_s or 0) / 60, cmd[0])
            proc.kill()
            rc = proc.wait()
        pumper.join(timeout=30)
        elapsed = time.monotonic() - start
        fh.write(f"\n--- exit {rc} ({describe_exit(rc)}) after {elapsed:.1f}s ---\n")

    tail_text = "".join(list(tail)[-tail_lines:])
    if timed_out or (check and rc != 0):
        raise CommandFailed(cmd, rc, tail_text, log_path, timed_out=timed_out, timeout_s=timeout_s)
    return rc, tail_text

## 4. Machine report

Recorded so a number in a CSV can never be separated from the box that produced
it. `env_metadata()` already stamps `gpu`, `torch` and `git_commit` onto every
results row; this is the same information, up front, in the log.

In [ ]:
def report_machine():
    log.info("python  %s", platform.python_version())
    log.info("os      %s", platform.platform())
    try:
        import torch

        log.info("torch   %s (cuda %s)", torch.__version__, torch.version.cuda)
        log.info("cuda available: %s", torch.cuda.is_available())
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                log.info(
                    "gpu %d   %s, %.1f GB, cc %d.%d",
                    i,
                    props.name,
                    props.total_memory / 1024**3,
                    props.major,
                    props.minor,
                )
    except Exception:
        log.exception("torch import failed")
    try:
        import numpy

        log.info("numpy   %s", numpy.__version__)
    except Exception:
        log.exception("numpy import failed")

    for label, path in (("work", WORK_DIR), ("out", OUT_DIR)):
        usage = shutil.disk_usage(path)
        log.info(
            "disk %-5s %.1f GB free of %.1f GB  (%s)",
            label,
            usage.free / 1024**3,
            usage.total / 1024**3,
            path,
        )
    try:
        rc, _ = sh("nvidia-smi", log_path=LOG_DIR / "setup.log", check=False)
        if rc != 0:
            log.warning("nvidia-smi returned %s -- is a GPU accelerator selected?", rc)
    except FileNotFoundError:
        log.error("nvidia-smi not found: this runtime has no GPU. Stop and enable one.")


report_machine()

## 5. Clone the repo

No `pip install -e .` of the project. Two reasons:

- `pyproject.toml` pins `requires-python >=3.11,<3.12`, which fails outright on
  a 3.12 image;
- its `torch` / `torchvision` entries resolve from PyPI on Linux (the CUDA index
  is scoped to `sys_platform == 'win32'`), which would replace the preinstalled
  CUDA build with a CPU wheel and quietly cost you the GPU.

Setting `PYTHONPATH` to the repo root plus `src/` imports both `bench` and
`spatial_ingestion` with none of that risk. Nothing in Tier B uses the
`zshift-*` console scripts.

In [ ]:
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    log.info("repo already present, fetching")
    sh(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "pull", "--ff-only"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", check=False)
else:
    sh(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "50", REPO_URL, str(REPO_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )

_, _commit = sh(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", echo=False
)
COMMIT = _commit.strip().splitlines()[-1] if _commit.strip() else "unknown"
log.info("repo at commit %s", COMMIT)

MAST3R_DIR = REPO_DIR / "third_party" / "mast3r"
DUST3R_DIR = MAST3R_DIR / "dust3r"

# Repo root for `bench`, src/ for `spatial_ingestion`.
PYTHONPATH = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "src")])

CHILD_ENV = {
    "PYTHONPATH": PYTHONPATH,
    "PYTHONUNBUFFERED": "1",
    "PYTHONFAULTHANDLER": "1",
    "HF_HOME": str(HF_HOME),
    "HUGGINGFACE_HUB_CACHE": str(HF_HOME / "hub"),
    "TOKENIZERS_PARALLELISM": "false",
    # PyVista/VTK are used for mesh filtering, never rendering, but this
    # runtime is headless -- make the intent explicit.
    "PYVISTA_OFF_SCREEN": "true",
    "MPLBACKEND": "Agg",
}

missing = [p for p in ("bench", "src/spatial_ingestion") if not (REPO_DIR / p).exists()]
if missing:
    raise SystemExit(
        f"missing from the clone: {missing}. These are untracked on the laptop -- "
        "commit and push them, then re-run."
    )
log.info("bench/ and src/spatial_ingestion/ present")

## 6. Dependencies

Read straight out of `pyproject.toml`, minus `torch`/`torchvision` (keep the
preinstalled CUDA build) and minus `numpy` unless you opted into the `<2` pin.

In [ ]:
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore

with open(REPO_DIR / "pyproject.toml", "rb") as fh:
    _pyproject = tomllib.load(fh)

SKIP_PREFIXES = ("torch", "torchvision")
deps = []
for dep in _pyproject["project"]["dependencies"]:
    name = dep.split(">")[0].split("<")[0].split("=")[0].split("[")[0].strip().lower()
    if name.startswith(SKIP_PREFIXES):
        log.info("skipping %-12s (keeping the preinstalled CUDA build)", name)
        continue
    if name == "numpy" and not PIN_NUMPY_LT2:
        log.info("skipping %-12s (set PIN_NUMPY_LT2 only if imports actually fail)", name)
        continue
    deps.append(dep)

log.info("installing %d dependencies", len(deps))
sh(
    [sys.executable, "-m", "pip", "install", "-q", *deps],
    log_path=LOG_DIR / "setup.log",
    timeout_s=1800,
)
log.info("dependencies installed")
if PIN_NUMPY_LT2:
    log.warning("numpy pinned <2 -- RESTART THE KERNEL now, then re-run from cell 2")

## 7. MASt3R

Mirrors `scripts/setup-mast3r.sh` (same pinned commit) but installs with
`--no-deps`: MASt3R's and DUSt3R's own `requirements.txt` list `torch`, and
letting pip resolve them is the other way to lose the CUDA build. Their real
dependencies are already in the project's list, which is why `roma`, `einops`,
`pyglet<2` and friends are there.

The RoPE CUDA kernel compile fails on your laptop and should succeed here. That
is a genuinely different code path from the one Tier A measured, and belongs in
the paper's setup section.

In [ ]:
PINNED_MAST3R = "f5209afc300cec36239a7ac992263f36847bbba0"

PY_STUB = """[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
version = "0.1.0"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["."]
include = ["{name}*"]
"""

if not MAST3R_DIR.exists():
    sh(
        ["git", "clone", "https://github.com/naver/mast3r", str(MAST3R_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    sh(["git", "checkout", PINNED_MAST3R], cwd=MAST3R_DIR, log_path=LOG_DIR / "setup.log")
    sh(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=MAST3R_DIR,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
else:
    log.info("mast3r already cloned at %s", MAST3R_DIR)

for target, name in ((MAST3R_DIR, "mast3r"), (DUST3R_DIR, "dust3r")):
    stub = target / "pyproject.toml"
    if not stub.exists():
        stub.write_text(PY_STUB.format(name=name), encoding="utf-8")
        log.info("wrote %s", stub)
    sh(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(target)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=900,
    )

# RoPE CUDA kernels: a speedup, not a requirement. Never fatal.
curope = DUST3R_DIR / "croco" / "models" / "curope"
try:
    sh(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=curope,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    log.info("RoPE CUDA kernels compiled")
    CUROPE_BUILT = True
except CommandFailed as exc:
    CUROPE_BUILT = False
    log.warning(
        "RoPE kernel compile failed (%s); falling back to the PyTorch path. "
        "Not fatal, but note it in the setup section. Log: %s",
        describe_exit(exc.returncode),
        exc.log_path,
    )

## 8. Scene manifest

Written into the work dir with **absolute** paths, so a read-only
`/kaggle/input` mount works and nothing has to be copied.

In [ ]:
MANIFEST_PATH = WORK_DIR / "scenes.json"
data_root = Path(DATA_ROOT)

manifest = {
    "_generated_by": "tier_b_gpu.ipynb run " + RUN_ID,
    "tau": 2.0,
    "units": "mm",
    "scenes": [
        {
            **scene,
            "image_dir": str((data_root / scene["image_dir"]).resolve()),
            "gt_path": (
                str((data_root / scene["gt_path"]).resolve()) if scene.get("gt_path") else None
            ),
        }
        for scene in SCENES
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
log.info("manifest -> %s (%d scenes)", MANIFEST_PATH, len(manifest["scenes"]))
print(MANIFEST_PATH.read_text())

## 9. Preflight

A hard gate. Everything that can fail cheaply fails here, before hours of
compute: imports, CUDA, checkpoint download, every scene path, every GT file,
frame counts, disk. Fatal problems raise; the rest are warnings you should read.

In [ ]:
FATAL = []
WARN = []


def check(label, fn, fatal=True):
    try:
        result = fn()
    except Exception as exc:
        (FATAL if fatal else WARN).append(f"{label}: {exc!r}")
        log.exception("FAIL  %s", label)
        return None
    log.info("ok    %-34s %s", label, "" if result is None else result)
    return result


def _imports():
    mods = (
        "torch",
        "numpy",
        "cv2",
        "trimesh",
        "pyvista",
        "scipy",
        "sklearn",
        "roma",
        "einops",
        "mast3r.model",
        "dust3r.utils.image",
        "spatial_ingestion.reconstruction.pipeline",
        "spatial_ingestion.final_pipeline.handoff",
        "bench.tier_b_common",
    )
    src = (
        "import importlib\n"
        + "\n".join(f"importlib.import_module({m!r})" for m in mods)
        + "\nprint('imports ok')"
    )
    sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        echo=False,
        timeout_s=600,
    )
    return f"{len(mods)} modules"


def _cuda():
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "torch.cuda.is_available() is False. Enable a GPU accelerator "
            "(Kaggle: Settings > Accelerator; Colab: Runtime > Change runtime type)."
        )
    free, total = torch.cuda.mem_get_info(0)
    name = torch.cuda.get_device_name(0)
    return f"{name}, {free / 1024**3:.1f}/{total / 1024**3:.1f} GB free"


def _checkpoint():
    # Download the weights here, where a network failure costs seconds.
    src = (
        "import time\n"
        "from mast3r.model import AsymmetricMASt3R\n"
        "t = time.time()\n"
        "m = AsymmetricMASt3R.from_pretrained("
        "'naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric')\n"
        "m = m.to('cuda').eval()\n"
        "n = sum(p.numel() for p in m.parameters())\n"
        "print('loaded %.0fM params to cuda in %.1fs' % (n / 1e6, time.time() - t))\n"
    )
    _, tail = sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        timeout_s=1800,
        echo=False,
    )
    lines = [ln for ln in tail.strip().splitlines() if ln.strip()]
    return lines[-1] if lines else "loaded"


def _scenes():
    entries = json.loads(MANIFEST_PATH.read_text())["scenes"]
    if not entries:
        raise RuntimeError("manifest has no scenes")
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    summary = []
    for entry in entries:
        image_dir = Path(entry["image_dir"])
        if not image_dir.is_dir():
            raise FileNotFoundError(f"{entry['name']}: image_dir does not exist: {image_dir}")
        n = sum(1 for p in image_dir.iterdir() if p.suffix.lower() in suffixes)
        if n == 0:
            raise FileNotFoundError(f"{entry['name']}: no images under {image_dir}")
        gt = entry.get("gt_path")
        if not gt:
            WARN.append(
                f"{entry['name']}: gt_path is null. B2/B4/B6/B8 skip scenes with no GT -- "
                "this scene contributes zero rows."
            )
        elif not Path(gt).exists():
            raise FileNotFoundError(f"{entry['name']}: gt_path missing: {gt}")
        if n <= 40:
            WARN.append(
                f"{entry['name']}: only {n} frames. B4 caps at MAX_RECONSTRUCTION_FRAMES=40, "
                "so the budget never fires and its three variants are identical."
            )
        summary.append(f"{entry['name']}={n} frames")
    return ", ".join(summary)


def _disk():
    problems = []
    for label, path, need_gb in (("work", WORK_DIR, 25), ("out", OUT_DIR, 2)):
        free = shutil.disk_usage(path).free / 1024**3
        if free < need_gb:
            problems.append(f"{label} has {free:.1f} GB free, want >= {need_gb} GB")
    if problems:
        raise RuntimeError("; ".join(problems))
    return "sufficient"


check("imports", _imports)
check("cuda", _cuda)
check("scenes", _scenes)
check("disk", _disk)
check("mast3r checkpoint", _checkpoint)

print()
for w in WARN:
    log.warning("WARN  %s", w)
if FATAL:
    for f in FATAL:
        log.error("FATAL %s", f)
    raise SystemExit(
        f"{len(FATAL)} preflight failure(s). Fix these before running anything long. "
        f"Full detail: {LOG_DIR / 'preflight.log'}"
    )
log.info("preflight passed (%d warnings)", len(WARN))

## 10. Experiment table

`--quick` is deliberately never passed: no `exp_b*.py` module forwards it to
`run()`, so it would do nothing while looking like it did something. Scope comes
from the flags each module actually reads.

Note B4: its frame budget is the module constant `DEFAULT_BUDGET = 40` with no
CLI override, so even the smoke lane runs three 40-frame reconstructions. It is
the longest smoke item by a wide margin, which is why it sits late in
`EXPERIMENTS`. Wiring `--quick` into `run()` is the real fix.

In [ ]:
MODULES = {
    "b1": ("b1_end_to_end", "bench.exp_b1_end_to_end"),
    "b2": ("b2_reconstruction_accuracy", "bench.exp_b2_reconstruction_accuracy"),
    "b3": ("b3_pairing_ablation", "bench.exp_b3_pairing_ablation"),
    "b4": ("b4_frame_budget_ablation", "bench.exp_b4_frame_budget_ablation"),
    "b5": ("b5_tsdf_fallback", "bench.exp_b5_tsdf_fallback"),
    "b6": ("b6_refinement_effect", "bench.exp_b6_refinement_effect"),
    "b7": ("b7_determinism", "bench.exp_b7_determinism"),
    "b8": ("b8_exif_intrinsics", "bench.exp_b8_exif_intrinsics"),
}

FIRST_SCENE = manifest["scenes"][0]["name"]

SMOKE_ARGS = {
    "b1": ["--scenes", FIRST_SCENE, "--no-rig"],
    "b2": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b3": ["--scenes", FIRST_SCENE, "--frame-counts", "8"],
    "b4": ["--scenes", FIRST_SCENE, "--no-budget-sweep"],
    "b5": ["--scenes", FIRST_SCENE, "--frame-counts", "8", "--thresholds", "0.2"],
    "b6": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b7": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b8": ["--scenes", FIRST_SCENE, "--n-images", "4"],
}

# Full lane: module defaults, every scene. Override here if the protocol changed.
FULL_ARGS = {key: [] for key in MODULES}

# b1 takes --deliverables-root; b2..b8 take --output-root. Both keep the
# ~1.5 GB per-run alignment cache off the persisted output volume.
OUTPUT_FLAG = {"b1": "--deliverables-root"}


def build_argv(exp, scope):
    _, module = MODULES[exp]
    args = (SMOKE_ARGS if scope == "smoke" else FULL_ARGS)[exp]
    out_root = WORK_DIR / "runs" / exp
    out_root.mkdir(parents=True, exist_ok=True)
    return [
        sys.executable,
        "-u",
        "-m",
        module,
        "--manifest",
        str(MANIFEST_PATH),
        "--results-dir",
        str(RESULTS_DIR),
        "--seed",
        "0",
        "--verbose",
        OUTPUT_FLAG.get(exp, "--output-root"),
        str(out_root),
        *args,
    ]


for exp in EXPERIMENTS:
    print(exp, " ".join(str(a) for a in build_argv(exp, SCOPE)[3:]))

## 11. Runner

One subprocess per experiment. A CUDA OOM, an OOM-killer `SIGKILL` or a VTK
segfault takes down that experiment only; the notebook records it and moves on.
`run_summary.json` is rewritten after each one, so a session killed at the wall
clock still leaves a complete account.

In [ ]:
SUMMARY_PATH = OUT_DIR / "run_summary.json"
RUNS = []
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")


def _save_summary():
    payload = {
        "run_id": RUN_ID,
        "platform": PLATFORM,
        "commit": COMMIT,
        "scope": SCOPE,
        "curope_built": CUROPE_BUILT,
        "started": SESSION_START_ISO,
        "updated": _dt.datetime.now().isoformat(timespec="seconds"),
        "runs": RUNS,
    }
    SUMMARY_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def _free_gb(path):
    return shutil.disk_usage(path).free / 1024**3


def _csv_rows(path):
    with path.open(encoding="utf-8") as fh:
        return max(0, sum(1 for _ in fh) - 1)


def run_experiment(exp, scope=None, force=None):
    scope = scope or SCOPE
    force = FORCE_RERUN if force is None else force
    exp_id, _ = MODULES[exp]
    csv_path = RESULTS_DIR / (exp_id + ".csv")
    log_path = LOG_DIR / (exp + ".log")

    if csv_path.exists() and not force:
        log.info("SKIP  %s -- %s exists (set FORCE_RERUN=True to redo)", exp, csv_path.name)
        record = {
            "exp": exp,
            "exp_id": exp_id,
            "status": "skipped",
            "minutes": 0.0,
            "rows": _csv_rows(csv_path),
            "csv": str(csv_path),
            "log": str(log_path),
        }
        RUNS.append(record)
        _save_summary()
        return record

    argv = build_argv(exp, scope)
    timeout_s = TIMEOUT_MIN[scope][exp] * 60
    log.info("=" * 70)
    log.info("START %s (%s lane, timeout %d min)", exp, scope, TIMEOUT_MIN[scope][exp])
    log.info("      disk free: work %.1f GB, out %.1f GB", _free_gb(WORK_DIR), _free_gb(OUT_DIR))
    log.info("      log -> %s", log_path)

    started = time.monotonic()
    record = {
        "exp": exp,
        "exp_id": exp_id,
        "scope": scope,
        "argv": [str(a) for a in argv],
        "started": _dt.datetime.now().isoformat(timespec="seconds"),
        "log": str(log_path),
    }
    try:
        rc, _tail = sh(
            argv,
            cwd=REPO_DIR,
            env=CHILD_ENV,
            log_path=log_path,
            timeout_s=timeout_s,
            check=True,
            echo=True,
        )
        record["status"] = "ok"
        record["returncode"] = rc
    except CommandFailed as exc:
        record["status"] = "timeout" if exc.timed_out else "failed"
        record["returncode"] = exc.returncode
        record["error"] = exc.reason
        record["tail"] = exc.tail
        log.error("FAIL  %s: %s", exp, exc.reason)
        log.error("      last lines:\n%s", exc.tail)
    except Exception as exc:  # the runner itself broke
        record["status"] = "error"
        record["error"] = repr(exc)
        record["traceback"] = traceback.format_exc()
        log.exception("ERROR %s: runner failed", exp)

    record["minutes"] = round((time.monotonic() - started) / 60, 2)
    if csv_path.exists():
        record["csv"] = str(csv_path)
        record["rows"] = _csv_rows(csv_path)
        log.info("      %s: %d rows -> %s", exp, record["rows"], csv_path.name)
        if record["rows"] == 0:
            log.warning("      %s produced ZERO rows -- check gt_path and --scenes", exp)
    else:
        record["csv"] = None
        record["rows"] = 0
        if record["status"] == "ok":
            log.warning("      %s exited 0 but wrote no CSV", exp)

    log.info("DONE  %s: %s in %.1f min", exp, record["status"], record["minutes"])
    RUNS.append(record)
    _save_summary()

    if CLEAN_WORK_AFTER_EACH:
        target = WORK_DIR / "runs" / exp
        if target.exists():
            size_gb = sum(f.stat().st_size for f in target.rglob("*") if f.is_file()) / 1024**3
            shutil.rmtree(target, ignore_errors=True)
            log.info("      cleaned %.1f GB from %s", size_gb, target)
    return record

## 12. Run the tier

Stops launching new experiments once `SESSION_BUDGET_MIN` is reached, rather
than starting something the platform's wall clock will cut off. Re-run the
notebook to pick up where it stopped.

In [ ]:
SESSION_START = time.monotonic()
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")
RUNS.clear()

log.info("Tier B: %s lane, %d experiments, commit %s", SCOPE, len(EXPERIMENTS), COMMIT[:8])

for exp in EXPERIMENTS:
    elapsed_min = (time.monotonic() - SESSION_START) / 60
    if elapsed_min > SESSION_BUDGET_MIN:
        log.warning(
            "session budget reached (%.0f min); not launching %s. Re-run the "
            "notebook to continue -- finished experiments are skipped.",
            elapsed_min,
            exp,
        )
        RUNS.append(
            {
                "exp": exp,
                "status": "not_started",
                "minutes": 0.0,
                "rows": 0,
                "reason": "session budget",
            }
        )
        _save_summary()
        continue
    run_experiment(exp)

print()
print("=" * 78)
print(f"{'exp':<5} {'status':<12} {'min':>7} {'rows':>6}  log")
print("-" * 78)
for r in RUNS:
    print(
        f"{r['exp']:<5} {r.get('status', '?'):<12} {r.get('minutes', 0.0):>7.1f} "
        f"{r.get('rows', 0):>6}  {Path(r.get('log', '-')).name}"
    )
print("=" * 78)

failed = [r for r in RUNS if r.get("status") in ("failed", "error", "timeout")]
print(f"\n{len(RUNS) - len(failed)}/{len(RUNS)} ok; summary -> {SUMMARY_PATH}")
for r in failed:
    print(f"\n--- {r['exp']}: {r.get('error', '')} ---")
    print(r.get("tail") or r.get("traceback") or "(see log)")

## 13. Collect results

Everything the paper needs, zipped into one file: the CSVs, every log, and the
run summary. On Kaggle the zip appears under the notebook's Output tab; on Colab
it lands in Drive if you mounted it.

In [ ]:
bundle = OUT_DIR / ("tier_b_" + RUN_ID)
bundle.mkdir(parents=True, exist_ok=True)
(bundle / "results").mkdir(exist_ok=True)

for csv in RESULTS_DIR.glob("*.csv"):
    shutil.copy2(csv, bundle / "results" / csv.name)
shutil.copytree(LOG_DIR, bundle / "logs", dirs_exist_ok=True)
if SUMMARY_PATH.exists():
    shutil.copy2(SUMMARY_PATH, bundle / "run_summary.json")

(bundle / "environment.txt").write_text(
    "\n".join(
        [
            "run_id      " + RUN_ID,
            "platform    " + PLATFORM,
            "commit      " + COMMIT,
            "scope       " + SCOPE,
            "python      " + platform.python_version(),
            "os          " + platform.platform(),
            "curope      " + ("compiled" if CUROPE_BUILT else "pytorch fallback"),
        ]
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
log.info("bundle -> %s (%.1f MB)", archive, Path(archive).stat().st_size / 1024**2)

print("\nCSVs produced:")
for csv in sorted((bundle / "results").glob("*.csv")):
    print(f"  {csv.name:<44} {_csv_rows(csv):>6} rows  {csv.stat().st_size / 1024:.0f} KB")

print("\nTo bring these home: download the zip, unpack into bench/results/, then")
print("  git add bench/results && git commit -m 'bench: Tier B results'")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `exit -9 / SIGKILL` | host RAM OOM killer, not VRAM | lower `--n-images`, or `--image-size 384` |
| status `timeout` | exceeded `TIMEOUT_MIN[scope][exp]` | raise the ceiling if the run was genuinely progressing; the log shows where it stalled |
| `torch.cuda.OutOfMemoryError` | VRAM | lower `--image-size`; for B3/B5 lower `--frame-counts` |
| `exit -11 / SIGSEGV` | native crash in VTK or CUDA | read `logs/<run_id>/faulthandler.log` for the Python frames |
| exits 0, **zero rows** | scene has `gt_path: null`, or `--scenes` matched nothing | scene names must match the manifest exactly |
| B4's three variants identical | scene has <= 40 frames, so the budget never fires | use a real extracted video sequence |
| `MASt3R is not installed` | cell 7 did not finish, or the kernel restarted | re-run cell 7; `--no-deps` means torch is never touched |
| `numpy.dtype size changed` | numpy 2 vs a wheel built for numpy 1 | set `PIN_NUMPY_LT2 = True`, restart the kernel, re-run from cell 2 |
| session killed at 12 h | platform wall clock | re-run; completed CSVs are skipped |
| `disk quota exceeded` on Kaggle | reconstructions landing in `/kaggle/working` | confirm `--output-root` points into `/kaggle/temp` |

**For the paper:** run every Tier B row on one GPU type. `env_metadata()` stamps
`gpu`, `torch` and `git_commit` onto every row, so mixed hardware is at least
detectable after the fact - but it is not comparable, and B7's determinism
result in particular is meaningless across devices.